# Phase 2 — Luke 2023 acquisition and data profiling

Approved historical dataset. This notebook verifies the real raw snapshot and displays the full-scan quality report. Counts are raw records/publisher labels, not final unique-posting KPIs. No cleaning, accepted classification, SQL, EDA or Tableau implementation occurs here.

In [1]:
from pathlib import Path
import json, sys
root=Path.cwd()
if not (root/'config/phase2_plan.json').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
from src.luke_dataset import read_json, validate_approval, verify_raw, verify_manifest
plan=read_json(root/'config/phase2_plan.json')
access=read_json(root/'config/source_access.json')
validate_approval(plan,access)
raw=root/plan['raw_directory']/plan['filename']
verify_raw(raw,plan)
manifest=read_json(raw.parent/'manifest.json')
verify_manifest(manifest,plan,raw)
profile=read_json(root/plan['profile_json'])
assert profile['raw_sha256']==manifest['sha256']
assert profile['raw_row_count']==manifest['row_count']
print(json.dumps({k:profile[k] for k in ['source','revision','raw_row_count','date_min','date_max','invalid_nonblank_dates']},indent=2))

{
  "source": "lukebarousse/data_jobs",
  "revision": "ed776e5a0a8c40ea9d5efbd800772ae52e140f3e",
  "raw_row_count": 785741,
  "date_min": "2023-01-01T00:00:04",
  "date_max": "2023-12-31T23:59:58",
  "invalid_nonblank_dates": 0
}


In [2]:
import pandas as pd
bay=profile['preliminary_bay_area']
print('Preliminary city-matched records:',bay['row_count'])
print('Counties represented:',len([n for n in bay['county_counts'].values() if n>0]))
display(pd.DataFrame.from_dict(bay['role_quality'],orient='index').rename_axis('publisher_role_label'))

Preliminary city-matched records: 15892
Counties represented: 9


,populated_skill_lists,positive_finite_annual_values,rows
publisher_role_label,,,
Business Analyst,449,29,523
Cloud Engineer,19,1,22
Data Analyst,3879,274,4640
Data Engineer,1641,178,1753
Data Scientist,4702,605,5557
Machine Learning Engineer,151,59,172
Senior Data Analyst,786,89,883
Senior Data Engineer,491,75,513
Senior Data Scientist,1404,224,1586


In [3]:
quality=pd.DataFrame.from_dict(bay['salary_fields'],orient='index')
display(quality)
print('Skill-list structural status:',bay['skill_list_status'])
print('Raw work-from-home flags:',bay['work_from_home_raw_values'])
print('All-source exact fingerprint duplicate candidates:',profile['exact_duplicate_row_candidates'])
print('Bay Area duplicate-key candidates:',bay['employer_title_location_date_duplicate_candidates'])

,infinite,invalid_numeric,nonpositive,numeric,present
salary_hour_avg,0,0,0,720,720
salary_year_avg,0,0,0,1584,1584


Skill-list structural status: {'missing': 2155, 'populated': 13737}
Raw work-from-home flags: {'False': 15892}
All-source exact fingerprint duplicate candidates: 101
Bay Area duplicate-key candidates: 5


## Interpret the evidence carefully

Exact city/CA matches are a preliminary geographic diagnostic. Original full text, native posting IDs, advertised salary bounds/currency/provenance are absent. Skill tags and role labels are inherited. A false work-from-home flag cannot identify on-site/hybrid jobs. Dates lack verified timezone and collection consistency. Salary numeric validity is not proof of advertised USD compensation.

To recompute the entire profile from the verified raw input, run `python -m src.luke_dataset profile` at the project root. The source file stays unchanged. See `reports/phase2_status.md`, `docs/phase2_field_dictionary.md` and the KPI amendment for metric availability. Phase 3 requires approval.